# Phase 5b — Nine named top-level groups

The top level of the bubbles is now a fixed set of 9 groups people can start from:
**Care, Guide, Protect, Build, Invent, Discover, Create, Connect, Organize**.

Because the names are chosen up front, plain clustering can't produce them. Instead this is **guided
(zero-shot) classification**:

1. Each group is described by ~6–8 example task statements, written in O*NET's style (`src/top_groups.py`).
2. The examples are embedded with the same model as the jobs (`all-mpnet-base-v2`); their average is the group's prototype.
3. Each job goes to its most similar prototype.
4. **Refinement:** a few rounds where each group's score blends the prototype similarity (weight `anchor`) with
   similarity to the centre of the jobs currently in the group, a seeded K-Means that lets the data sharpen
   boundaries without losing what each name means.

5. **Fixed rule:** all transportation jobs (SOC 53-2 to 53-6, plus transportation supervisors) and delivery and
   logistics jobs (mail carriers, couriers, logisticians, freight agents, dispatchers…) go to **Connect**
   ("keep people and goods moving"). Applied after steps 1–4 by `apply_rules()`.

The subclusters inside each group are still found unsupervised (HDBSCAN, Phase 5).

In [1]:
import sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "../src")
import numpy as np, pandas as pd
from sklearn.metrics import adjusted_rand_score
from top_groups import GROUPS, GROUP_NAMES, prototypes, assign, apply_rules, load_embeddings
from load_onet import PROCESSED
pd.set_option("display.max_colwidth", 120)

X, codes = load_embeddings()
P = prototypes()
titles = pd.read_csv(PROCESSED / "occupations.csv", index_col="code").loc[codes, "title"].values
interest = pd.read_csv(PROCESSED / "interests.csv", index_col="code").loc[codes, "top_interest"].values
soc = codes.str[:2].values
names = np.array(GROUP_NAMES)
pd.DataFrame([(g["name"], g["tagline"], len(g["examples"])) for g in GROUPS], columns=["group", "tagline", "examples"])

,group,tagline,examples
0,Care,I want to care for people's health and wellbeing,6
1,Guide,"I want to teach, counsel and help people grow",6
2,Protect,"I want to keep people, places and rights safe",6
3,Build,"I want to build, fix and operate things with my hands",6
4,Invent,I want to design and engineer new technology,6
5,Discover,"I want to research, analyze and understand how things work",6
6,Create,"I want to make art, media and designs",7
7,Connect,"I want to serve the public, sell, and keep people and goods moving",11
8,Organize,"I want to manage, plan and keep things running",6


## 1. Choosing the refinement strength
`anchor` = how much weight the group descriptions keep during refinement (1.0 = no refinement).

Besides the overall numbers, we check a set of jobs where the right answer is clear.

In [2]:
expected = {
    "Registered Nurses": "Care", "Preschool Teachers, Except Special Education": "Guide", "Lawyers": "Protect",
    "Firefighters": "Protect", "Electricians": "Build", "Airline Pilots, Copilots, and Flight Engineers": "Connect",
    "Bus Drivers, Transit and Intercity": "Connect", "Software Developers": "Invent", "Mechanical Engineers": "Invent",
    "Chemists": "Discover", "Graphic Designers": "Create", "Chefs and Head Cooks": "Connect",
    "Retail Salespersons": "Connect", "Accountants and Auditors": "Organize", "Chief Executives": "Organize",
}
rows = []
for anchor in [0.5, 0.6, 0.7, 0.8, 1.0]:
    l = assign(X, P, refine_rounds=8, anchor=anchor)
    g = pd.Series(apply_rules(pd.Series(names[l], index=codes)).values, index=titles)
    wrong = [f"{t.split(',')[0]} → {g[t]}" for t, want in expected.items() if g[t] != want]
    rows.append({"anchor": anchor, "RIASEC_ARI": round(adjusted_rand_score(interest, g.values), 3),
                 "SOC_ARI": round(adjusted_rand_score(soc, g.values), 3), "checks_passed": f"{len(expected) - len(wrong)}/{len(expected)}",
                 "misplaced": "; ".join(wrong)})
pd.DataFrame(rows)

,anchor,RIASEC_ARI,SOC_ARI,checks_passed,misplaced
0,0.5,0.355,0.361,15/15,
1,0.6,0.364,0.359,15/15,
2,0.7,0.363,0.350,15/15,
3,0.8,0.367,0.347,15/15,
4,1.0,0.367,0.335,15/15,


- Transportation is placed by a fixed rule, so pilots and bus drivers pass at every setting. Before the rule,
  weak anchoring let them drift into **Protect** (their tasks are full of safety language).
- See the table for which setting passes all checks; **0.7** is the default in `top_groups()`.

For comparison, the 6 hybrid clusters had RIASEC ARI 0.35, and numeric-only clusters 0.24.

## 2. The final groups

In [3]:
labels = assign(X, P, refine_rounds=8, anchor=0.7)
groups = apply_rules(pd.Series(names[labels], index=codes))
summary = pd.DataFrame({
    "jobs": groups.value_counts(),
    "main interest": pd.Series(interest, index=codes).groupby(groups).agg(lambda s: s.value_counts().index[0]),
    "examples": groups.groupby(groups).apply(lambda s: "; ".join(pd.Series(titles, index=codes)[s.index].sample(5, random_state=0))),
}).loc[GROUP_NAMES]
summary

,jobs,main interest,examples
Care,121,Social,Obstetricians and Gynecologists; Personal Care Aides; Nursing Assistants; Mental Health and Substance Abuse Social W...
Guide,80,Social,"Preschool Teachers, Except Special Education; Anthropology and Archeology Teachers, Postsecondary; Geography Teacher..."
Protect,54,Conventional,"Construction and Building Inspectors; Bailiffs; Lifeguards, Ski Patrol, and Other Recreational Protective Service Wo..."
Build,243,Realistic,"Industrial Machinery Mechanics; Grinding, Lapping, Polishing, and Buffing Machine Tool Setters, Operators, and Tende..."
Invent,90,Realistic,Natural Sciences Managers; Computer Programmers; Robotics Engineers; Validation Engineers; Nanotechnology Engineerin...
Discover,77,Investigative,"Sociologists; Animal Breeders; Environmental Science and Protection Technicians, Including Health; Agricultural Tech..."
Create,22,Artistic,"Makeup Artists, Theatrical and Performance; Choreographers; Writers and Authors; News Analysts, Reporters, and Journ..."
Connect,104,Realistic,"Food Preparation Workers; Couriers and Messengers; Transportation, Storage, and Distribution Managers; Dispatchers, ..."
Organize,132,Conventional,"Switchboard Operators, Including Answering Service; Set and Exhibit Designers; Compliance Managers; Financial Manage..."


## 3. Sanity check against O*NET's job families
Rows are official SOC major groups; columns are our groups. Most families land mainly in one group,
and the splits that remain are reasonable (e.g. Management splits across Organize, Connect and Guide by what is managed).

In [4]:
soc_names = {"11":"Management","13":"Business/finance","15":"Computer","17":"Engineering","19":"Science",
             "21":"Social service","23":"Legal","25":"Education","27":"Arts/media","29":"Health practitioners",
             "31":"Health support","33":"Protective","35":"Food","37":"Cleaning","39":"Personal care",
             "41":"Sales","43":"Office","45":"Farming","47":"Construction","49":"Repair","51":"Production","53":"Transport"}
pd.crosstab(pd.Series(soc).map(soc_names).rename("SOC family"), groups.values)[GROUP_NAMES]

col_0,Care,Guide,Protect,Build,Invent,Discover,Create,Connect,Organize
SOC family,,,,,,,,,
Arts/media,0,2,1,1,4,0,20,5,7
Business/finance,0,1,6,1,0,4,0,9,27
Cleaning,0,0,1,5,0,0,0,0,2
Computer,1,0,3,0,24,4,0,0,4
Construction,0,0,1,57,2,1,0,0,0
Education,0,57,0,1,0,0,0,0,4
Engineering,0,0,2,1,45,8,0,0,0
Farming,0,0,0,8,0,3,0,0,1
Food,0,0,0,0,0,0,0,15,1


## 4. Notes
- **Create is small (~22 jobs)** because O*NET has few arts occupations and several (e.g. editors, broadcast technicians)
  are closer to other groups. That reflects the data, not a bug.
- Example sentences matter: e.g. a retail example in Connect fixed misplaced salespeople. Transportation is
  handled by a fixed rule instead, so it can't drift. Edit `GROUPS` in `src/top_groups.py` to adjust; the embeddings re-cache automatically.